<a href="https://colab.research.google.com/github/zara-nasiruddin/computational-art-history/blob/main/project-01-renaissance-mannerism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from datasets import load_dataset
from collections import Counter

dataset = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
)

selected_artists = {
    "high_renaissance": {
        "train": [
            "michelangelo", "giorgione", "dosso-dossi",
            "pietro-perugino", "lorenzo-lotto",
            "luca-signorelli"
        ],
        "test": ["raphael", "leonardo-da-vinci"]
    },
    "mannerism_late_renaissance": {
        "train": [
            "hans-von-aachen", "maerten-van-heemskerck",
            "giorgio-vasari", "jacopo-pontormo",
            "el-greco", "tintoretto"
        ],
        "test": ["parmigianino", "agnolo-bronzino"]
    }
}

# Build a lookup of the artists selected for each style.
artists_by_style = {
    style: set(group["train"] + group["test"])
    for style, group in selected_artists.items()
}

counts = Counter()

# Read metadata only; no need to load painting images yet.
metadata = dataset.select_columns(["artist", "style"])

for item in metadata:
    artist = str(item["artist"]).lower().strip()
    style = str(item["style"]).lower().strip()

    if style in artists_by_style and artist in artists_by_style[style]:
        counts[(style, artist)] += 1

print("PAINTING COUNTS")
for style, groups in selected_artists.items():
    print(f"\n{style.upper()}")
    for split_name, artists in groups.items():
        for artist in artists:
            print(f"{split_name:5} | {artist:25} | {counts[(style, artist)]}")


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

PAINTING COUNTS

HIGH_RENAISSANCE
train | michelangelo              | 173
train | giorgione                 | 48
train | dosso-dossi               | 40
train | pietro-perugino           | 122
train | lorenzo-lotto             | 107
train | luca-signorelli           | 61
test  | raphael                   | 185
test  | leonardo-da-vinci         | 152

MANNERISM_LATE_RENAISSANCE
train | hans-von-aachen           | 41
train | maerten-van-heemskerck    | 63
train | giorgio-vasari            | 72
train | jacopo-pontormo           | 103
train | el-greco                  | 162
train | tintoretto                | 227
test  | parmigianino              | 46
test  | agnolo-bronzino           | 82


In [ ]:

import random

random.seed(42)

eligible = {
    "high_renaissance": [],
    "mannerism_late_renaissance": []
}

for (style, artist), count in counts.items():
    if count >= 40:
        eligible[style].append(artist)

# Exclude artists appearing in both categories
overlap = (
    set(eligible["high_renaissance"])
    & set(eligible["mannerism_late_renaissance"])
)

for style in eligible:
    eligible[style] = [
        artist for artist in eligible[style]
        if artist not in overlap
    ]

split = {}

for style, artists in eligible.items():
    chosen = sorted(artists)
    random.shuffle(chosen)
    chosen = chosen[:8]

    split[style] = {
        "train": chosen[:6],
        "test": chosen[6:]
    }

    print(f"\n{style}")
    print("Training artists:", split[style]["train"])
    print("Testing artists:", split[style]["test"])

print("\nArtist overlap check:")

train_artists = set()
test_artists = set()

for style in split:
    train_artists.update(split[style]["train"])
    test_artists.update(split[style]["test"])

print("Artists shared between train and test:",
      sorted(train_artists & test_artists))



high_renaissance
Training artists: ['michelangelo', 'giorgione', 'dosso-dossi', 'pietro-perugino', 'lorenzo-lotto', 'luca-signorelli']
Testing artists: ['raphael', 'leonardo-da-vinci']

mannerism_late_renaissance
Training artists: ['hans-von-aachen', 'maerten-van-heemskerck', 'giorgio-vasari', 'jacopo-pontormo', 'el-greco', 'tintoretto']
Testing artists: ['parmigianino', 'agnolo-bronzino']

Artist overlap check:
Artists shared between train and test: []


In [ ]:

from collections import defaultdict

artists_by_style = defaultdict(set)

for (style, artist), count in counts.items():
    if count > 0:
        artists_by_style[style].add(artist)

high = artists_by_style["high_renaissance"]
mannerism = artists_by_style["mannerism_late_renaissance"]

overlap = sorted(high & mannerism)

print("Artists appearing in BOTH categories:")
print(overlap)

print("\nHigh Renaissance artists exclusive to that category:")
for artist, count in sorted(
    [
        (artist, n)
        for (style, artist), n in counts.items()
        if style == "high_renaissance"
        and artist not in overlap
        and n >= 20
    ],
    key=lambda x: x[1],
    reverse=True
):
    print(f"{artist}: {count}")

print("\nMannerism artists exclusive to that category:")
for artist, count in sorted(
    [
        (artist, n)
        for (style, artist), n in counts.items()
        if style == "mannerism_late_renaissance"
        and artist not in overlap
        and n >= 20
    ],
    key=lambda x: x[1],
    reverse=True
):
    print(f"{artist}: {count}")


Artists appearing in BOTH categories:
['andrea-del-sarto', 'correggio', 'titian']

High Renaissance artists exclusive to that category:
raphael: 185
michelangelo: 173
leonardo-da-vinci: 152
pietro-perugino: 122
lorenzo-lotto: 107
andrea-mantegna: 69
luca-signorelli: 61
vittore-carpaccio: 52
giorgione: 48
dosso-dossi: 40
giovanni-bellini: 32
cima-da-conegliano: 28
pinturicchio: 23

Mannerism artists exclusive to that category:
tintoretto: 227
paolo-veronese: 183
el-greco: 162
jacopo-pontormo: 103
agnolo-bronzino: 82
giorgio-vasari: 72
maerten-van-heemskerck: 63
parmigianino: 46
hans-von-aachen: 41
giuseppe-arcimboldo: 25
sofonisba-anguissola: 21
piero-di-cosimo: 20


In [ ]:

from collections import Counter
from datasets import load_dataset

dataset = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
).select_columns(["artist", "style"])

counts = Counter()

for item in dataset:
    style = str(item["style"]).lower().strip()
    artist = str(item["artist"]).lower().strip()

    if style in [
        "high_renaissance",
        "mannerism_late_renaissance"
    ]:
        counts[(style, artist)] += 1

for style in [
    "high_renaissance",
    "mannerism_late_renaissance"
]:
    print(f"\n--- {style} ---")
    artists = [
        (artist, count)
        for (artist_style, artist), count in counts.items()
        if artist_style == style
    ]

    for artist, count in sorted(
        artists, key=lambda x: x[1], reverse=True
    ):
        print(f"{artist}: {count} paintings")


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]


--- high_renaissance ---
raphael: 185 paintings
michelangelo: 173 paintings
leonardo-da-vinci: 152 paintings
titian: 131 paintings
pietro-perugino: 122 paintings
lorenzo-lotto: 107 paintings
andrea-mantegna: 69 paintings
luca-signorelli: 61 paintings
vittore-carpaccio: 52 paintings
giorgione: 48 paintings
andrea-del-sarto: 47 paintings
dosso-dossi: 40 paintings
correggio: 32 paintings
giovanni-bellini: 32 paintings
cima-da-conegliano: 28 paintings
pinturicchio: 23 paintings
andrea-solario: 19 paintings
giovanni-antonio-boltraffio: 16 paintings

--- mannerism_late_renaissance ---
tintoretto: 227 paintings
paolo-veronese: 183 paintings
el-greco: 162 paintings
titian: 121 paintings
jacopo-pontormo: 103 paintings
agnolo-bronzino: 82 paintings
giorgio-vasari: 72 paintings
maerten-van-heemskerck: 63 paintings
parmigianino: 46 paintings
hans-von-aachen: 41 paintings
correggio: 28 paintings
giuseppe-arcimboldo: 25 paintings
sofonisba-anguissola: 21 paintings
piero-di-cosimo: 20 paintings
ross

In [ ]:

from collections import Counter
from datasets import load_dataset

dataset = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
)

metadata = dataset.select_columns(
    ["image_id", "artist", "style", "date"]
)

style_counts = Counter()
target_artists = {
    "high_renaissance": set(),
    "mannerism_late_renaissance": set()
}

print("Checking style labels and artist names...")

for item in metadata:
    style = str(item["style"]).lower().strip()
    artist = str(item["artist"]).lower().strip()

    style_counts[style] += 1

    if style in target_artists:
        target_artists[style].add(artist)

print("\nTotal records checked:", sum(style_counts.values()))

print("\nRenaissance-related styles:")
for style, count in sorted(style_counts.items()):
    if "renaissance" in style or "mannerism" in style:
        print(f"{style}: {count}")

print("\nDistinct artists in our target categories:")
for style, artists in target_artists.items():
    print(f"{style}: {len(artists)}")


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Checking style labels and artist names...

Total records checked: 74234

Renaissance-related styles:
early_renaissance: 1382
high_renaissance: 1337
mannerism_late_renaissance: 1267
northern_renaissance: 2535

Distinct artists in our target categories:
high_renaissance: 18
mannerism_late_renaissance: 19


In [ ]:

from datasets import load_dataset

dataset = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
)

print("Dataset connection successful!")

for item in dataset:
    print("\nFields available:")
    print(list(item.keys()))

    print("\nFirst painting record:")
    for key, value in item.items():
        if isinstance(value, str):
            print(f"{key}: {value[:300]}")
        else:
            print(f"{key}: {type(value).__name__}")
    break


README.md:   0%|          | 0.00/5.53k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Dataset connection successful!

Fields available:
['image_id', 'title', 'artist', 'style', 'date', 'e_narrative', 'e_formal', 'e_emotional', 'e_historical', 'dominant_emotion', 'artemis_coverage', 'rag_sim', 'n_perspectives', 'e_unified', 'image']

First painting record:
image_id: Abstract_Expressionism/aaron-siskind_acolman-1-1955.jpg
title: aaron-siskind_acolman-1-1955.jpg
artist: aaron-siskind
style: abstract_expressionism
date: float
e_narrative: Ancient stone pillars stand sentinel, weathered and silent, casting deep shadows that whisper of forgotten times. Nature creeps in—grasses and vines reclaiming the ruins. Sunlight filters through unseen gaps, illuminating textures of decay and endurance. The scene breathes with stillness, as if hold
e_formal: The painting depicts a desolate and atmospheric landscape. The composition is dominated by the towering, crumbling walls of a medieval abbey, rendered in a muted palette of grays and browns. The walls are depicted in various states of

In [ ]:

import datasets

print("Success! The datasets library is installed.")
print("Version:", datasets.__version__)



Success! The datasets library is installed.
Version: 4.8.5


In [ ]:

import urllib.request
import json

url = "https://api.github.com/repos/asahi417/wikiart-image-dataset/contents/wikiartcrawler/groups"

request = urllib.request.Request(
    url,
    headers={"User-Agent": "Research-Experiment"}
)

with urllib.request.urlopen(request, timeout=20) as response:
    files = json.loads(response.read().decode("utf-8"))

print("Available group files:\n")

for item in files:
    print(item["name"])


Available group files:

__init__.py
abstract-expressionism.txt
abstract_expressionism.py
baroque.py
baroque.txt
ecole-de-paris.txt
ecole_de_paris.py
expressionism.py
expressionism.txt
impressionism.py
impressionism.txt
naive-art-primitivism.txt
naive_art_primitivism.py
neo-impressionism.txt
neo_impressionism.py
neoclassicism.py
neoclassicism.txt
post-impressionism.txt
post_impressionism.py
pre-raphaelite-brotherhood.txt
pre_raphaelite_brotherhood.py
realism.py
realism.txt
rococo.py
rococo.txt
romanticism.py
romanticism.txt
surrealism.py
surrealism.txt
symbolism.py
symbolism.txt


In [ ]:

import urllib.request
import json

url = "https://api.github.com/repos/asahi417/wikiart-image-dataset/contents/wikiartcrawler"

request = urllib.request.Request(
    url,
    headers={"User-Agent": "Research-Experiment"}
)

with urllib.request.urlopen(request, timeout=20) as response:
    files = json.loads(response.read().decode("utf-8"))

print("Files inside wikiartcrawler:\n")

for item in files:
    print(item["name"], "-", item["type"])


Files inside wikiartcrawler:

__init__.py - file
artist_group.py - file
groups - dir
util.py - file
wikiart_api.py - file


In [ ]:

import urllib.request

url = "https://raw.githubusercontent.com/asahi417/wikiart-image-dataset/master/README.md"

request = urllib.request.Request(
    url,
    headers={"User-Agent": "Research-Experiment"}
)

with urllib.request.urlopen(request, timeout=20) as response:
    readme = response.read().decode("utf-8")

print(readme[:12000])


# WikiART Face/WikiART General Imageset & WikiART Crawler


<p align="center">
<img src="assets/wikiart_face_image.png" width="650"><br>
<em>Figure 1: Samples from WikiART Face dataset.</em>
</p>

***News:*** WikiART Face is available in huggingface. Check the repo at [asahi417/wikiart_face](https://huggingface.co/datasets/asahi417/wikiart_face).

This is a repository for [***WikiART Crawler***](#wikiart-crawler), a python-library to download/process images from
[WikiART](https://www.wikiart.org/) via WikiART API. We also have released following two image datasets created 
with the WikiART Crawler.

- [***WikiART Face***](#wikiart-face): Facial image dataset extracted from the portraits obtained via WikiART Crawler. 
- [***WikiART General***](#wikiart-general): Image dataset of paintings (landscape and portrait) obtained via WikiART Crawler.

The datasets can be used via [WikiART Crawler](#wikiart-crawler) that allows fine-grained control of the dataset. 
Or check [dataset links sectio

In [ ]:

import urllib.request
import json

url = "https://api.github.com/repos/asahi417/wikiart-image-dataset/contents"

request = urllib.request.Request(
    url,
    headers={"User-Agent": "Research-Experiment"}
)

with urllib.request.urlopen(request, timeout=20) as response:
    files = json.loads(response.read().decode("utf-8"))

print("Files in the repository:\n")

for item in files:
    print(item["name"], "-", item["type"])


Files in the repository:

.gitattributes - file
.gitignore - file
README.md - file
assets - dir
examples - dir
setup.cfg - file
setup.py - file
tests - dir
wikiartcrawler - dir


In [ ]:

import urllib.request
import json

url = "https://api.github.com/repos/asahi417/wikiart-image-dataset"

request = urllib.request.Request(
    url,
    headers={"User-Agent": "Research-Experiment"}
)

with urllib.request.urlopen(request, timeout=20) as response:
    repo = json.loads(response.read().decode("utf-8"))

print("Repository:", repo["full_name"])
print("Description:", repo["description"])
print("Last updated:", repo["updated_at"])
print("Default branch:", repo["default_branch"])
print("Repository page:", repo["html_url"])


Repository: asahi417/wikiart-image-dataset
Description: We release WikiART Crawler, a python-library to download/process images from WikiART via WikiART API, and two image datasets: `WikiART Face` and `WikiART General`.
Last updated: 2026-08-31T18:59:12Z
Default branch: master
Repository page: https://github.com/asahi417/wikiart-image-dataset


In [ ]:

import urllib.request
import json

url = "https://api.github.com/search/repositories?q=WikiArt+dataset"

request = urllib.request.Request(
    url,
    headers={"User-Agent": "Research-Experiment"}
)

with urllib.request.urlopen(request, timeout=20) as response:
    data = json.loads(response.read().decode("utf-8"))

print("Repositories found:", data["total_count"])

for repo in data["items"][:5]:
    print("\nRepository:", repo["full_name"])
    print("Description:", repo.get("description"))
    print("URL:", repo["html_url"])


Repositories found: 75

Repository: asahi417/wikiart-image-dataset
Description: We release WikiART Crawler, a python-library to download/process images from WikiART via WikiART API, and two image datasets: `WikiART Face` and `WikiART General`.
URL: https://github.com/asahi417/wikiart-image-dataset

Repository: mbellitti/wikiart-classifier
Description: Teaching a Convolutional Neural Network to recognize painting genre. Handcrafted dataset. Cool visualizations.
URL: https://github.com/mbellitti/wikiart-classifier

Repository: annahung31/wikiart_dataset
Description: This is a repo for using wikiart dataset
URL: https://github.com/annahung31/wikiart_dataset

Repository: robbiebarrat/wikiart-dataset
Description: huge corpus of paintings ready for use in machine learning
URL: https://github.com/robbiebarrat/wikiart-dataset

Repository: bhargavvader/WikiArtVectors
Description: Repository for the WikiArtVectors dataset, where we provide style and color representations for 68094 artworks from 

In [ ]:

import re

# Look for image links and references to our two art styles
image_links = re.findall(
    r'https?://[^"\']+\.(?:jpg|jpeg|png|webp)(?:\?[^"\']*)?',
    html,
    flags=re.IGNORECASE
)

print("Image links found:", len(set(image_links)))
print("High Renaissance mentions:", html.lower().count("high renaissance"))
print("Mannerism mentions:", html.lower().count("mannerism"))

print("\nExample image links:")
for link in list(dict.fromkeys(image_links))[:5]:
    print(link[:180])


Image links found: 38
High Renaissance mentions: 1
Mannerism mentions: 3

Example image links:
https://uploads.wikiart.org/Content/wiki/img/favicon-256x256.png
https://uploads.wikiart.org/Content/wiki/img/search.svg) center no-repeat;background-size:24px 24px;width:68px;height:100%;position:absolute;right:0;top:0;border-right:none;padding
https://uploads.wikiart.org/Content/wiki/img/auth-icon@2x.png
https://uploads.wikiart.org/Content/wiki/img/logo_small.png
https://uploads.wikiart.org/Content/wiki/img/arrow_down.png


In [ ]:

import urllib.request
from html.parser import HTMLParser

url = "https://www.wikiart.org/"
request = urllib.request.Request(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
)

with urllib.request.urlopen(request, timeout=15) as response:
    html = response.read().decode("utf-8", errors="replace")

class TitleParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_title = False
        self.title = ""

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "title":
            self.in_title = True

    def handle_data(self, data):
        if self.in_title:
            self.title += data

    def handle_endtag(self, tag):
        if tag.lower() == "title":
            self.in_title = False

parser = TitleParser()
parser.feed(html)

print("Page title:", parser.title.strip())
print("HTML characters received:", len(html))
print("Contains Renaissance:", "renaissance" in html.lower())



Page title: WikiArt.org - Visual Art Encyclopedia
HTML characters received: 388273
Contains Renaissance: True


In [ ]:

import urllib.request

url = "https://www.wikiart.org/"
request = urllib.request.Request(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
)

try:
    response = urllib.request.urlopen(request, timeout=15)
    print("WikiArt response:", response.status)
except Exception as e:
    print("WikiArt access:", e)



WikiArt response: 200


In [ ]:

import urllib.request

url = "https://www.wikiart.org/"

try:
    response = urllib.request.urlopen(url, timeout=15)
    print("WikiArt website is reachable!")
    print("Response code:", response.status)
except Exception as e:
    print("Website check:", e)


Website check: HTTP Error 403: Forbidden


In [ ]:

import urllib.request

url = "https://www.google.com"

try:
    response = urllib.request.urlopen(url, timeout=10)
    print("Internet access: Working")
    print("Response code:", response.status)
except Exception as e:
    print("Internet access check failed:")
    print(e)


Internet access: Working
Response code: 200


In [ ]:
import sys
print("python version:", sys.version)

python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [ ]:
print("High Renaissance vc Mannerism")
print(" our reserach experiment starts here!")

High Renaissance vc Mannerism
 our reserach experiment starts here!


In [ ]:
import torch
import transformers
import sklearn
from datasets import load_dataset

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Dataset library: ready")
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.11.0+cpu
Transformers: 5.18.0
Scikit-learn: 1.6.1
Dataset library: ready
GPU available: False


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
)

print("Available columns:", dataset.column_names)

first_item = next(iter(dataset))
print("First record keys:", first_item.keys())

for key, value in first_item.items():
    if key != "image":
        print(f"{key}: {str(value)[:200]}")
    else:
        print("image:", type(value))

README.md:   0%|          | 0.00/5.53k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Available columns: ['image_id', 'title', 'artist', 'style', 'date', 'e_narrative', 'e_formal', 'e_emotional', 'e_historical', 'dominant_emotion', 'artemis_coverage', 'rag_sim', 'n_perspectives', 'e_unified', 'image']
First record keys: dict_keys(['image_id', 'title', 'artist', 'style', 'date', 'e_narrative', 'e_formal', 'e_emotional', 'e_historical', 'dominant_emotion', 'artemis_coverage', 'rag_sim', 'n_perspectives', 'e_unified', 'image'])
image_id: Abstract_Expressionism/aaron-siskind_acolman-1-1955.jpg
title: aaron-siskind_acolman-1-1955.jpg
artist: aaron-siskind
style: abstract_expressionism
date: 1955.0
e_narrative: Ancient stone pillars stand sentinel, weathered and silent, casting deep shadows that whisper of forgotten times. Nature creeps in—grasses and vines reclaiming the ruins. Sunlight filters through unse
e_formal: The painting depicts a desolate and atmospheric landscape. The composition is dominated by the towering, crumbling walls of a medieval abbey, rendered in a mute

In [ ]:

from collections import defaultdict
import random
from datasets import load_dataset

random.seed(42)
N_PER_ARTIST = 20

train_artists = {
    "michelangelo", "giorgione", "dosso-dossi",
    "pietro-perugino", "lorenzo-lotto", "luca-signorelli",
    "hans-von-aachen", "maerten-van-heemskerck",
    "giorgio-vasari", "jacopo-pontormo", "el-greco", "tintoretto"
}

test_artists = {
    "raphael", "leonardo-da-vinci",
    "parmigianino", "agnolo-bronzino"
}

artist_style = {
    artist: "high_renaissance" for artist in {
        "michelangelo", "giorgione", "dosso-dossi",
        "pietro-perugino", "lorenzo-lotto", "luca-signorelli",
        "raphael", "leonardo-da-vinci"
    }
}
artist_style.update({
    artist: "mannerism_late_renaissance" for artist in {
        "hans-von-aachen", "maerten-van-heemskerck",
        "giorgio-vasari", "jacopo-pontormo", "el-greco",
        "tintoretto", "parmigianino", "agnolo-bronzino"
    }
})

targets = train_artists | test_artists
samples = defaultdict(list)
seen = defaultdict(int)

dataset = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
).select_columns(["artist", "style", "image"])

for item in dataset:
    artist = str(item["artist"]).lower().strip()
    style = str(item["style"]).lower().strip()

    if artist not in targets or style != artist_style[artist]:
        continue

    key = (style, artist)
    seen[key] += 1
    image = item["image"]

    if len(samples[key]) < N_PER_ARTIST:
        samples[key].append(image)
    else:
        j = random.randrange(seen[key])
        if j < N_PER_ARTIST:
            samples[key][j] = image

print("Sample selection complete.")
print("Artists collected:", len(samples))
print("Total paintings selected:", sum(map(len, samples.values())))

for (style, artist), images in sorted(samples.items()):
    print(style, "|", artist, "|", len(images))


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107327830 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (99962094 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
'peer closed connection without sending complete message body (received 205942487 bytes, expected 226389168)' thrown while requesting GET https://huggingface.co/datasets/Shuaimyself/MMArt/resolve/ee0626506aa9f90a557d6a7548d32ef88dd40779/data/train-00014-of-00022.parquet
Retrying in 1s [Retry 1/5].


Sample selection complete.
Artists collected: 16
Total paintings selected: 320
high_renaissance | dosso-dossi | 20
high_renaissance | giorgione | 20
high_renaissance | leonardo-da-vinci | 20
high_renaissance | lorenzo-lotto | 20
high_renaissance | luca-signorelli | 20
high_renaissance | michelangelo | 20
high_renaissance | pietro-perugino | 20
high_renaissance | raphael | 20
mannerism_late_renaissance | agnolo-bronzino | 20
mannerism_late_renaissance | el-greco | 20
mannerism_late_renaissance | giorgio-vasari | 20
mannerism_late_renaissance | hans-von-aachen | 20
mannerism_late_renaissance | jacopo-pontormo | 20
mannerism_late_renaissance | maerten-van-heemskerck | 20
mannerism_late_renaissance | parmigianino | 20
mannerism_late_renaissance | tintoretto | 20


In [ ]:

import torch
from transformers import CLIPModel, CLIPProcessor

model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)
model.eval()

# Take two paintings from each movement
pilot_images = []

for (style, artist), images in samples.items():
    if style == "high_renaissance" and len(pilot_images) < 2:
        pilot_images.append(images[0].convert("RGB"))
        if len(pilot_images) == 2:
            break

for (style, artist), images in samples.items():
    if style == "mannerism_late_renaissance":
        pilot_images.append(images[0].convert("RGB"))
        if len(pilot_images) == 4:
            break

inputs = processor(
    images=pilot_images,
    return_tensors="pt"
)

with torch.no_grad():
    features = model.get_image_features(**inputs)

print("Number of images:", len(pilot_images))
print("Embedding shape:", tuple(features.shape))
print("All embeddings finite:", torch.isfinite(features).all().item())


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

NameError: name 'samples' is not defined

In [ ]:
print("samples exists:", "samples" in globals())
print("model exists:", "model" in globals())

samples exists: False
model exists: True


In [ ]:

import torch

pilot_images = []

for (style, artist), images in samples.items():
    if style == "high_renaissance" and len(pilot_images) < 2:
        pilot_images.append(images[0].convert("RGB"))

for (style, artist), images in samples.items():
    if style == "mannerism_late_renaissance" and len(pilot_images) < 4:
        pilot_images.append(images[0].convert("RGB"))

inputs = processor(images=pilot_images, return_tensors="pt")

with torch.no_grad():
    features = model.get_image_features(**inputs)

print("Number of images:", len(pilot_images))
print("Embedding shape:", tuple(features.shape))
print("All embeddings finite:", torch.isfinite(features).all().item())



Number of images: 4


AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'shape'

In [ ]:

import torch

print("Returned object:", type(features))
print("Available fields:", features.keys())
print("Output shape:", features.pooler_output.shape)


Returned object: <class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
Available fields: odict_keys(['last_hidden_state', 'pooler_output'])
Output shape: torch.Size([4, 512])


In [ ]:

import torch

features = features.pooler_output

print("Number of images:", len(pilot_images))
print("Embedding shape:", tuple(features.shape))
print("All embeddings finite:", torch.isfinite(features).all().item())


Number of images: 4
Embedding shape: (4, 512)
All embeddings finite: True


In [ ]:

import torch
import numpy as np

embedding_records = []
failed_images = []

model.eval()

with torch.no_grad():
    for (style, artist), images in sorted(samples.items()):
        for image_index, image in enumerate(images):
            try:
                image = image.convert("RGB")
                inputs = processor(images=image, return_tensors="pt")

                # Extract the image representation
                output = model.vision_model(
                    pixel_values=inputs["pixel_values"]
                )

                # Convert the visual representation to 512 features
                features = model.visual_projection(output.pooler_output)

                embedding = features[0].cpu().numpy()

                if not np.isfinite(embedding).all():
                    raise ValueError("Embedding contains invalid values")

                embedding_records.append({
                    "artist": artist,
                    "style": style,
                    "image_index": image_index,
                    "embedding": embedding
                })

            except Exception as e:
                failed_images.append({
                    "artist": artist,
                    "image_index": image_index,
                    "error": str(e)
                })

print("Successful paintings:", len(embedding_records))
print("Failed paintings:", len(failed_images))
print("Feature vector shape:",
      embedding_records[0]["embedding"].shape
      if embedding_records else "No embeddings")


Successful paintings: 320
Failed paintings: 0
Feature vector shape: (512,)


In [ ]:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Build the feature matrix and labels
X = np.stack([
    record["embedding"] for record in embedding_records
])

y = np.array([
    record["style"] for record in embedding_records
])

artists = np.array([
    record["artist"] for record in embedding_records
])

# Artists reserved exclusively for testing
test_artists = {
    "raphael",
    "leonardo-da-vinci",
    "parmigianino",
    "agnolo-bronzino"
}

train_mask = ~np.isin(artists, list(test_artists))
test_mask = np.isin(artists, list(test_artists))

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("Training paintings:", len(X_train))
print("Testing paintings:", len(X_test))
print("Training artists:", len(set(artists[train_mask])))
print("Testing artists:", len(set(artists[test_mask])))
print("Training class counts:",
      {s: int(np.sum(y_train == s)) for s in np.unique(y_train)})
print("Testing class counts:",
      {s: int(np.sum(y_test == s)) for s in np.unique(y_test)})

# Train a simple classifier
classifier = LogisticRegression(
    max_iter=2000,
    random_state=42
)
classifier.fit(X_train, y_train)

# Evaluate on unseen artists
predictions = classifier.predict(X_test)

print("\nAccuracy:", round(accuracy_score(y_test, predictions), 3))
print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, predictions), 3))
print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(
    y_test,
    predictions,
    labels=["high_renaissance", "mannerism_late_renaissance"]
))


Training paintings: 240
Testing paintings: 80
Training artists: 12
Testing artists: 4
Training class counts: {np.str_('high_renaissance'): 120, np.str_('mannerism_late_renaissance'): 120}
Testing class counts: {np.str_('high_renaissance'): 40, np.str_('mannerism_late_renaissance'): 40}

Accuracy: 0.562
Balanced accuracy: 0.562

Classification report:
                            precision    recall  f1-score   support

          high_renaissance       0.56      0.55      0.56        40
mannerism_late_renaissance       0.56      0.57      0.57        40

                  accuracy                           0.56        80
                 macro avg       0.56      0.56      0.56        80
              weighted avg       0.56      0.56      0.56        80

Confusion matrix:
[[22 18]
 [17 23]]


In [ ]:

from sklearn.metrics import accuracy_score
import numpy as np

test_artists_actual = artists[test_mask]

print("RESULTS BY UNSEEN ARTIST\n")

for artist in sorted(set(test_artists_actual)):
    artist_mask = test_artists_actual == artist

    actual = y_test[artist_mask]
    predicted = predictions[artist_mask]

    correct = int(np.sum(actual == predicted))
    total = len(actual)
    movement = actual[0]

    print(f"Artist: {artist}")
    print(f"Movement: {movement}")
    print(f"Correct: {correct}/{total}")
    print(f"Accuracy: {accuracy_score(actual, predicted):.1%}")
    print()


RESULTS BY UNSEEN ARTIST

Artist: agnolo-bronzino
Movement: mannerism_late_renaissance
Correct: 7/20
Accuracy: 35.0%

Artist: leonardo-da-vinci
Movement: high_renaissance
Correct: 7/20
Accuracy: 35.0%

Artist: parmigianino
Movement: mannerism_late_renaissance
Correct: 16/20
Accuracy: 80.0%

Artist: raphael
Movement: high_renaissance
Correct: 15/20
Accuracy: 75.0%



In [ ]:

from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Evaluate on the paintings used to train the classifier
train_predictions = classifier.predict(X_train)

print("TRAINING RESULTS")
print("Accuracy:", round(accuracy_score(y_train, train_predictions), 3))
print("Balanced accuracy:",
      round(balanced_accuracy_score(y_train, train_predictions), 3))

print("\nUNSEEN-ARTIST TEST RESULTS")
print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, predictions), 3))


TRAINING RESULTS
Accuracy: 0.996
Balanced accuracy: 0.996

UNSEEN-ARTIST TEST RESULTS
Accuracy: 0.562
Balanced accuracy: 0.562
